# Gold: Flight Features (Training Dataset)
**One row per flight** with all features joined + engineered:
  - Flight basics (airline, route, schedule)
  - Origin airport metadata (size, elevation, country)
  - Destination airport metadata
  - NOTAMs active at departure time (count + max severity)
  - Weather (placeholder — joined later when weather data exists)
  - **Target**: `dep_delay_minutes_computed`


**Source**: `silver.flights`, `silver.airports`, `silver.notams`   
**Target**: `gold.flight_features`

## Imports and Setup

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    abfss, SILVER_CONTAINER, GOLD_CONTAINER,
    configure_storage_auth,
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load Silver Tables

In [0]:
GOLD_FEATURES_PATH = abfss(GOLD_CONTAINER, "flight_features")

SILVER_FLIGHTS_PATH = abfss(SILVER_CONTAINER, "flights")
SILVER_AIRPORTS_PATH = abfss(SILVER_CONTAINER, "airports")
SILVER_NOTAMS_PATH = abfss(SILVER_CONTAINER, "notams")

configure_storage_auth(spark)

flights = spark.read.format("delta").load(SILVER_FLIGHTS_PATH)
airports = spark.read.format("delta").load(SILVER_AIRPORTS_PATH)
notams = spark.read.format("delta").load(SILVER_NOTAMS_PATH)


## Filter flight to trainable subset

In [0]:
flights_trainable = (
    flights
    # Must have a target variable
    .filter(F.col("dep_delay_minutes_computed").isNotNull())
    # Flight must actually have occurred (or at least been attempted)
    .filter(F.col("flight_status").isin("active", "landed"))
)


## Join: Departure airport metadata

In [0]:
airports_dep = airports.select(
    F.col("iata_code").alias("dep_iata"),
    F.col("airport_type").alias("dep_airport_type"),
    F.col("airport_size_rank").alias("dep_airport_size_rank"),
    F.col("latitude").alias("dep_latitude"),
    F.col("longitude").alias("dep_longitude"),
    F.col("elevation_ft").alias("dep_elevation_ft"),
    F.col("country_code").alias("dep_country"),
    F.col("city").alias("dep_city"),
)

flights_with_dep = flights_trainable.join(airports_dep, on="dep_iata", how="left")

## Join: Arrival airport metadata

In [0]:
airports_arr = airports.select(
    F.col("iata_code").alias("arr_iata"),
    F.col("airport_type").alias("arr_airport_type"),
    F.col("airport_size_rank").alias("arr_airport_size_rank"),
    F.col("latitude").alias("arr_latitude"),
    F.col("longitude").alias("arr_longitude"),
    F.col("elevation_ft").alias("arr_elevation_ft"),
    F.col("country_code").alias("arr_country"),
    F.col("city").alias("arr_city"),
)

flights_with_airports = flights_with_dep.join(airports_arr, on="arr_iata", how="left")

## Feature: Great-circle distance (km)

In [0]:

# Haversine formula via Spark 
R_EARTH_KM = 6371.0

flights_with_distance = (
    flights_with_airports
    .withColumn("dep_lat_rad", F.radians("dep_latitude"))
    .withColumn("arr_lat_rad", F.radians("arr_latitude"))
    .withColumn("dlat", F.radians(F.col("arr_latitude") - F.col("dep_latitude")))
    .withColumn("dlon", F.radians(F.col("arr_longitude") - F.col("dep_longitude")))
    .withColumn(
        "a",
        F.sin(F.col("dlat") / 2) ** 2
        + F.cos(F.col("dep_lat_rad")) * F.cos(F.col("arr_lat_rad"))
        * F.sin(F.col("dlon") / 2) ** 2
    )
    .withColumn(
        "route_distance_km",
        (2 * R_EARTH_KM * F.asin(F.sqrt("a"))).cast("int")
    )
    .drop("dep_lat_rad", "arr_lat_rad", "dlat", "dlon", "a")
)



## Feature: NOTAM activity at departure airport
For each flight, count NOTAMs active at the departure airport at `dep_scheduled_ts`, and extract max severity.

Aggregate NOTAMs per airport + time window. A NOTAM is "active" for a flight if:
- notam.start <= flight.dep_scheduled <= notam.end
- AND notam.location corresponds to flight.dep_icao


In [0]:

# Severity score for ranking
severity_score = (
    F.when(F.col("severity") == "critical", 4)
     .when(F.col("severity") == "high", 3)
     .when(F.col("severity") == "medium", 2)
     .when(F.col("severity") == "low", 1)
     .otherwise(0)
)

notams_scored = notams.withColumn("severity_score", severity_score)

# Join flights with notams where notam is active at dep time
# NOTE: using ICAO code for NOTAM join 
notam_join_condition = (
    (F.col("dep_icao") == F.col("notam_location"))
    & (F.col("start_ts_utc") <= F.col("dep_scheduled_ts"))
    & (F.col("end_ts_utc") >= F.col("dep_scheduled_ts"))
    # notam has to be known BEFORE the scheduled time to avoid DATA LEAKAGE :3
    & (F.col("start_ts_utc") <= F.col("dep_scheduled_ts") - F.expr("INTERVAL 2 HOURS"))
)

flights_with_notams = (
    flights_with_distance.alias("f")
    .join(
        notams_scored.alias("n"),
        on=notam_join_condition,
        how="left",
    )
    .groupBy(*[f"f.{c}" for c in flights_with_distance.columns])
    .agg(
        F.count("n.notam_number").alias("dep_notam_count"),
        F.max("n.severity_score").alias("dep_notam_max_severity_score"),
        F.sum(F.when(F.col("n.severity") == "critical", 1).otherwise(0)).alias("dep_notam_critical_count"),
        F.sum(F.when(F.col("n.severity") == "high", 1).otherwise(0)).alias("dep_notam_high_count"),
    )
)

# Rename the `f.<col>` prefix back to just <col> for readability
for old_col in flights_with_distance.columns:
    flights_with_notams = flights_with_notams.withColumnRenamed(f"f.{old_col}", old_col)


## Additional Time-based Features

In [0]:
gold = (
    flights_with_notams
    .withColumn("dep_day_of_week", F.dayofweek("dep_scheduled_ts"))      # 1=Sun … 7=Sat
    .withColumn("dep_is_weekend",
                F.col("dep_day_of_week").isin(1, 7).cast("int"))
    .withColumn("dep_hour_of_day", F.hour("dep_scheduled_ts"))
    .withColumn(
        "dep_time_of_day",
        F.when(F.col("dep_hour_of_day").between(5, 11), "morning")
         .when(F.col("dep_hour_of_day").between(12, 16), "afternoon")
         .when(F.col("dep_hour_of_day").between(17, 21), "evening")
         .otherwise("night")
    )
    .withColumn("dep_month", F.month("dep_scheduled_ts"))
    .withColumn(
        "scheduled_flight_duration_min",
        (
            (F.unix_timestamp("arr_scheduled_ts") - F.unix_timestamp("dep_scheduled_ts")) / 60
        ).cast("int")
    )
    # Target variable, clearly named for the ML teammate
    .withColumnRenamed("dep_delay_minutes_computed", "target_dep_delay_minutes")
    # Binary version of target (delayed = >15 min, industry standard)
    .withColumn(
        "target_is_delayed_15min",
        (F.col("target_dep_delay_minutes") > 15).cast("int")
    )
    .withColumn("gold_processed_ts_utc", F.current_timestamp())
)

## Select Final Features

In [0]:
gold_final = gold.select(
    # ---- Identifiers (not features, but useful for debugging / grouping) ----
    "flight_iata_number",
    "dep_scheduled_ts",
    "dep_scheduled_date_partition",

    # ---- Flight ----
    "airline_iata",
    "airline_name",
    "flight_type",           # departure / arrival
    "flight_status",

    # ---- Route ----
    "dep_iata",
    "dep_icao",
    "arr_iata",
    "arr_icao",
    "route_distance_km",
    "scheduled_flight_duration_min",

    # ---- Departure airport features ----
    "dep_airport_type",
    "dep_airport_size_rank",
    "dep_elevation_ft",
    "dep_country",
    "dep_city",
    "dep_latitude",
    "dep_longitude",

    # ---- Arrival airport features ----
    "arr_airport_type",
    "arr_airport_size_rank",
    "arr_elevation_ft",
    "arr_country",
    "arr_city",
    "arr_latitude",
    "arr_longitude",

    # ---- Time features ----
    "dep_day_of_week",
    "dep_is_weekend",
    "dep_hour_of_day",
    "dep_time_of_day",
    "dep_month",

    # ---- NOTAM features ----
    "dep_notam_count",
    "dep_notam_max_severity_score",
    "dep_notam_critical_count",
    "dep_notam_high_count",

    # ---- Target ----
    "target_dep_delay_minutes",
    "target_is_delayed_15min",

    # ---- Metadata ----
    "gold_processed_ts_utc",
)

## Write to Gold Delta Table

In [0]:
(
    gold_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("dep_scheduled_date_partition")
    .save(GOLD_FEATURES_PATH)
)

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS gold.flight_features
    USING DELTA
    LOCATION '{GOLD_FEATURES_PATH}'
""")

## Gold Data Summary

In [0]:

print(f"Total rows: {gold_final.count():,}")

print("\nTarget distribution")
display(gold_final.groupBy("target_is_delayed_15min").count())

print("\nDelay stats")
display(gold_final.agg(
    F.mean("target_dep_delay_minutes").alias("mean_delay"),
    F.expr("percentile(target_dep_delay_minutes, 0.5)").alias("median_delay"),
    F.min("target_dep_delay_minutes").alias("min_delay"),
    F.max("target_dep_delay_minutes").alias("max_delay"),
))

print("\nBy departure airport")
display(
    gold_final
    .groupBy("dep_iata")
    .agg(
        F.count("*").alias("flights"),
        F.mean("target_dep_delay_minutes").alias("avg_delay_min"),
        F.mean("target_is_delayed_15min").alias("pct_delayed"),
    )
    .orderBy(F.desc("flights"))
)

## Exit

In [0]:
dbutils.notebook.exit(f'{{"rows_in_gold": {gold_final.count()}}}')